In [0]:
# Purpose: Configure Bronze, Silver, and Quarantine paths.
# Safety: Configuration only; no data is modified.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "bronze/telemetry_stream/"
)

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry/"
)

quarantine_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/quarantine/telemetry/"
)

bronze_df = spark.read.format("delta").load(bronze_path)

print(f"Bronze records loaded: {bronze_df.count():,}")

Bronze records loaded: 41,522


In [0]:
# Purpose: Validate required fields and separate invalid records.
# Safety: Reads Bronze only; prepares DataFrames without writing output.

validated_df = bronze_df.withColumn(
    "event_timestamp_utc",
    F.to_timestamp("event_timestamp")
)

invalid_condition = (
    F.col("event_id").isNull()
    | (F.trim(F.col("event_id")) == "")
    | F.col("vehicle_id").isNull()
    | (F.trim(F.col("vehicle_id")) == "")
    | F.col("event_timestamp_utc").isNull()
    | (
        F.col("latitude").isNotNull()
        & ~F.col("latitude").between(-90, 90)
    )
    | (
        F.col("longitude").isNotNull()
        & ~F.col("longitude").between(-180, 180)
    )
    | (F.col("speed_kph") < 0)
    | (F.col("odometer_km") < 0)
    | (F.col("rpm") < 0)
    | (F.col("fuel_rate_lph") < 0)
)

invalid_df = validated_df.filter(invalid_condition)
valid_df = validated_df.filter(~invalid_condition)

print(f"Valid records: {valid_df.count():,}")
print(f"Invalid records: {invalid_df.count():,}")

Valid records: 41,510
Invalid records: 12


In [0]:
# Purpose: Inspect timestamp-related column types before writing to Delta.
# Safety: Read-only inspection; does not modify Bronze, Silver, or Quarantine.

print("=== Bronze schema ===")
bronze_df.select(
    "event_timestamp",
    "event_timestamp_utc",
    "bronze_ingested_at"
).printSchema()

print("=== Validated schema ===")
valid_df.select(
    "event_timestamp",
    "event_timestamp_utc",
    "bronze_ingested_at"
).printSchema()

=== Bronze schema ===
root
 |-- event_timestamp: string (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)

=== Validated schema ===
root
 |-- event_timestamp: string (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)



In [0]:
# Purpose: Check whether the Silver and Quarantine paths already contain Delta tables.
# Safety: Checks table metadata only; does not overwrite or delete any data.

from delta.tables import DeltaTable

print("Silver path is Delta:", DeltaTable.isDeltaTable(spark, silver_path))
print("Quarantine path is Delta:", DeltaTable.isDeltaTable(spark, quarantine_path))

Silver path is Delta: True
Quarantine path is Delta: False


In [0]:
# Purpose: Inspect the existing Silver Delta table schema and record count.
# Safety: Read-only; does not overwrite or delete existing Silver data.

silver_existing_df = spark.read.format("delta").load(silver_path)

print("=== Existing Silver schema ===")
silver_existing_df.printSchema()

print(f"Existing Silver records: {silver_existing_df.count():,}")

=== Existing Silver schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

Existing Silver records: 10,000,000


In [0]:
# Purpose: Define separate Silver output paths for streaming data.
# Safety: Keeps the existing 10-million-record Silver table untouched.

from pyspark.sql import functions as F
from delta.tables import DeltaTable

silver_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/telemetry_stream/"
)

quarantine_path = (
    "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"
    "silver/quarantine/telemetry_stream/"
)

print("Streaming Silver path:", silver_path)
print("Quarantine path:", quarantine_path)

print(
    "Streaming Silver already exists:",
    DeltaTable.isDeltaTable(spark, silver_path)
)

Streaming Silver path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/telemetry_stream/
Quarantine path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/silver/quarantine/telemetry_stream/
Streaming Silver already exists: False


In [0]:
# Purpose: Align streaming records with a consistent Silver schema.
# Safety: Transforms DataFrames in memory only; does not write or modify existing tables.

silver_ready_df = (
    valid_df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumnRenamed("bronze_ingested_at", "ingested_at")
    .withColumn("source_file", F.lit(None).cast("string"))
    .drop("event_timestamp_utc", "source_format", "event_json",
          "capture_sequence_number", "capture_enqueued_time_utc")
    .dropDuplicates(["event_id"])
)

silver_ready_df.printSchema()
print(f"Records ready for Silver: {silver_ready_df.count():,}")

root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

Records ready for Silver: 41,510


In [0]:
# Purpose: Write validated streaming telemetry to a new Silver Delta location.
# Safety: Check that the destination does not already contain a Delta table.
# The existing historical Silver table remains untouched.

from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, silver_path):
    raise RuntimeError(
        "Streaming Silver already exists. Stop and inspect it before writing."
    )

(
    silver_ready_df.write
    .format("delta")
    .mode("error")
    .save(silver_path)
)

print("Streaming Silver write completed successfully.")

Streaming Silver write completed successfully.


In [0]:
# Purpose: Verify the number of records written to Streaming Silver.
# Safety: Read-only verification; does not modify stored data.

silver_stream_df = spark.read.format("delta").load(silver_path)

print(f"Streaming Silver records: {silver_stream_df.count():,}")
silver_stream_df.printSchema()

Streaming Silver records: 41,510
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- speed_kph: double (nullable = true)
 |-- odometer_km: double (nullable = true)
 |-- rpm: long (nullable = true)
 |-- fuel_rate_lph: double (nullable = true)
 |-- engine_temp_c: double (nullable = true)
 |-- ignition_on: boolean (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- cargo_temp_c: double (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
# Purpose: Persist invalid telemetry records in a separate Quarantine Delta path.
# Safety: Refuse to write if a Quarantine Delta table already exists.

from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, quarantine_path):
    raise RuntimeError(
        "Quarantine already exists. Inspect it before writing."
    )

(
    invalid_df.write
    .format("delta")
    .mode("error")
    .save(quarantine_path)
)

print("Quarantine write completed successfully.")

Quarantine write completed successfully.


In [0]:
# Purpose: Verify the number of quarantined records.
# Safety: Read-only verification.

quarantine_df = spark.read.format("delta").load(quarantine_path)

print(f"Quarantined records: {quarantine_df.count():,}")

Quarantined records: 12
